# Data exploration and Feature Engineering

In [1]:
import os
import pandas as pd
import sqlite3

## EDA

In [2]:
gen_df = pd.read_csv("../data/processed/flat_files/SOLAR-GENERATION-TRENDS-2015-2026.csv")
capacity_df = pd.read_csv("../data/processed/flat_files/INSTALLED-CAPACITY-TRENDS-2015-2026.csv")
demand_df = pd.read_csv("../data/processed/flat_files/ENERGY-DEMAND-TRENDS-2015-2026.csv")
weather_df = pd.read_csv('../data/processed/flat_files/WEATHER-DATA-2015-2026.csv')

In [3]:
%load_ext sql
%sql sqlite:///../data/processed/databases/energy_master.db 

Tip: You may define configurations in /home/marvin/Workspace/Solar-Curtailment-Project/pyproject.toml or /home/marvin/.jupysql/config.

Did not find user configurations in /home/marvin/Workspace/Solar-Curtailment-Project/pyproject.toml.

Connecting to 'sqlite:///../data/processed/databases/energy_master.db'

In [4]:
con = sqlite3.connect('../data/processed/databases/energy_master.db')
cur = con.cursor()

In [5]:
gen_df.to_sql('GENERATION', con=con, index=False, if_exists='replace', method='multi')
capacity_df.to_sql('CAPACITY', con=con, index=False, if_exists='replace', method='multi')
demand_df.to_sql('DEMAND', con=con, index=False, if_exists='replace', method='multi')
weather_df.to_sql('WEATHER', con=con, index=False, if_exists='replace', method='multi')

4752

In [6]:
%%sql master <<
SELECT
    g.state,
    g.fyear,
    g.month_num,
    g.month_name,
    g.generation_mu,
    c.capacity_mw,
    d.peak_demand_mw,
    w.solar_irradiance_kwh_m2_day,
    w.temperature_2m_c
FROM GENERATION AS g
JOIN CAPACITY   AS c USING (state, fyear)
JOIN DEMAND     AS d USING (state, fyear, month_num)
JOIN WEATHER    AS w USING (state, fyear, month_num);

Running query in 'sqlite:///../data/processed/databases/energy_master.db'

In [7]:
master_df = master.DataFrame()
master_df

,state,fyear,month_num,month_name,generation_mu,capacity_mw,peak_demand_mw,solar_irradiance_kwh_m2_day,temperature_2m_c
0,Andhra Pradesh,2015-16,4,April,15.44,588.84,6794.0,6.4596,31.19
1,Andhra Pradesh,2015-16,8,August,12.15,588.84,6500.0,5.1350,29.22
2,Andhra Pradesh,2015-16,12,December,31.29,588.84,6544.0,4.5396,24.17
3,Andhra Pradesh,2015-16,2,February,40.32,588.84,7123.0,5.6628,27.60
4,Andhra Pradesh,2015-16,1,January,35.30,588.84,6868.0,4.5756,24.90
...,...,...,...,...,...,...,...,...,...
4370,West Bengal,2025-26,6,June,32.16,320.62,13108.0,4.5168,30.07
4371,West Bengal,2025-26,5,May,40.31,320.62,12693.0,5.6410,31.23
4372,West Bengal,2025-26,11,November,35.34,320.62,9070.0,4.1496,20.32
4373,West Bengal,2025-26,10,October,32.25,320.62,10283.0,4.3255,26.40


In [8]:
# %sql COMMIT 


In [9]:
%sql --close sqlite:///../data/processed/databases/energy_master.db
cur.close()